# Analisis exploratorio AquaLimpia S. A.

Este notebook documenta el flujo completo de analisis del dataset de aguas residuales: carga de datos, revision de calidad, construccion de indicadores, deteccion de valores atipicos, generacion de archivos de salida y visualizacion de resultados.

El objetivo es dejar una version reproducible del analisis que complemente los scripts del proyecto (`main.py` y modulos en `src/`).

## Hallazgos principales

- El dataset contiene 200 registros entre el 2025-07-01 y el 2025-10-28.
- La tasa general de cumplimiento normativo es 22.50%, por lo que la mayor parte de los registros queda clasificada como no cumple.
- La eficiencia promedio de remocion de DBO es 87.09%, pero existen 155 registros con alerta operativa por incumplimiento normativo o eficiencia menor a 70%.
- Planta Norte presenta la menor tasa de cumplimiento (16.90%) y la mayor cantidad de alertas junto con Planta Centro.
- Las anomalias por z-score aparecen principalmente en `lodos_generados_kg_d` y `DBO_salida_mg_L`.

In [ ]:
from pathlib import Path
import sys

ROOT_DIR = Path.cwd()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent

SRC_DIR = ROOT_DIR / "src"
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

DATA_PATH = ROOT_DIR / "data" / "dataset_set_A_aguas_residuales.xlsx"
OUTPUT_PATH = ROOT_DIR / "outputs"
OUTPUT_PATH.mkdir(exist_ok=True)

print(f"Directorio del proyecto: {ROOT_DIR}")
print(f"Dataset: {DATA_PATH}")
print(f"Salidas: {OUTPUT_PATH}")

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px

from src.procesamiento import cargar_datos
from src.calidad_datos import evaluar_calidad_datos
from src.indicadores import construir_indicadores, detectar_valores_atipicos
from main import generar_archivos_salida

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.2f}".format)

## 1. Carga de datos

Se carga el archivo Excel original y se convierte `fecha_registro` a tipo fecha. Esta etapa usa la misma funcion `cargar_datos` del modulo `src/procesamiento.py`.

In [ ]:
df_original = cargar_datos(DATA_PATH)

print(f"Filas: {df_original.shape[0]}")
print(f"Columnas: {df_original.shape[1]}")
print(f"Periodo: {df_original['fecha_registro'].min().date()} a {df_original['fecha_registro'].max().date()}")

df_original.head()

In [ ]:
df_original.info()

## 2. Calidad de datos

Se revisan tipos de datos, valores nulos y registros duplicados. El resumen se guarda en `outputs/calidad_datos.csv`.

In [ ]:
resumen_calidad = evaluar_calidad_datos(df_original, OUTPUT_PATH)
resumen_calidad

In [ ]:
duplicados = df_original.duplicated().sum()
nulos_totales = int(df_original.isna().sum().sum())

pd.DataFrame({
    "metrica": ["registros", "columnas", "valores_nulos", "duplicados"],
    "valor": [len(df_original), df_original.shape[1], nulos_totales, duplicados]
})

## 3. Exploracion inicial

Se revisa la distribucion de registros por planta y el comportamiento general de las variables numericas antes de construir indicadores derivados.

In [ ]:
registros_por_planta = (
    df_original["planta"]
    .value_counts()
    .rename_axis("planta")
    .reset_index(name="registros")
)

registros_por_planta

In [ ]:
fig = px.bar(
    registros_por_planta,
    x="planta",
    y="registros",
    text="registros",
    title="Registros disponibles por planta"
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="Cantidad de registros", xaxis_title="Planta")
fig.show()

In [ ]:
columnas_numericas_base = [
    "caudal_entrada_m3_d",
    "DBO_entrada_mg_L",
    "SST_entrada_mg_L",
    "pH_entrada",
    "energia_aeracion_kWh",
    "lodos_generados_kg_d",
    "DBO_salida_mg_L",
]

df_original[columnas_numericas_base].describe().T

## 4. Indicadores operativos y ambientales

Se calculan tres variables clave:

- `eficiencia_remocion_DBO_%`: porcentaje de remocion de DBO.
- `estado_cumplimiento`: etiqueta legible para el cumplimiento normativo.
- `alerta_operativa`: marca registros con incumplimiento normativo o eficiencia menor a 70%.

In [ ]:
df = construir_indicadores(df_original)
df = detectar_valores_atipicos(df)

df[[
    "fecha_registro",
    "planta",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_remocion_DBO_%",
    "cumplimiento_norma",
    "estado_cumplimiento",
    "alerta_operativa",
]].head()

In [ ]:
kpis = pd.DataFrame({
    "metrica": [
        "registros_analizados",
        "cumplimiento_normativo_%",
        "DBO_salida_promedio_mg_L",
        "eficiencia_promedio_DBO_%",
        "registros_con_alerta",
        "porcentaje_alertas_%",
    ],
    "valor": [
        len(df),
        round(df["cumplimiento_norma"].mean() * 100, 2),
        round(df["DBO_salida_mg_L"].mean(), 2),
        round(df["eficiencia_remocion_DBO_%"].mean(), 2),
        int((df["alerta_operativa"] == "Alerta").sum()),
        round((df["alerta_operativa"] == "Alerta").mean() * 100, 2),
    ]
})

kpis

## 5. Resumen por planta

El resumen permite comparar desempeno entre plantas y priorizar focos de revision operativa.

In [ ]:
resumen_planta = df.groupby("planta").agg(
    registros=("planta", "count"),
    caudal_promedio=("caudal_entrada_m3_d", "mean"),
    DBO_entrada_promedio=("DBO_entrada_mg_L", "mean"),
    DBO_salida_promedio=("DBO_salida_mg_L", "mean"),
    eficiencia_promedio=("eficiencia_remocion_DBO_%", "mean"),
    energia_promedio=("energia_aeracion_kWh", "mean"),
    lodos_promedio=("lodos_generados_kg_d", "mean"),
    tasa_cumplimiento=("cumplimiento_norma", "mean"),
    alertas=("alerta_operativa", lambda serie: int((serie == "Alerta").sum())),
).reset_index()

resumen_planta["tasa_cumplimiento"] = (resumen_planta["tasa_cumplimiento"] * 100).round(2)
resumen_planta["eficiencia_promedio"] = resumen_planta["eficiencia_promedio"].round(2)
resumen_planta["DBO_salida_promedio"] = resumen_planta["DBO_salida_promedio"].round(2)
resumen_planta["caudal_promedio"] = resumen_planta["caudal_promedio"].round(2)
resumen_planta["energia_promedio"] = resumen_planta["energia_promedio"].round(2)
resumen_planta["lodos_promedio"] = resumen_planta["lodos_promedio"].round(2)

resumen_planta.sort_values("tasa_cumplimiento")

In [ ]:
fig = px.bar(
    resumen_planta.sort_values("tasa_cumplimiento"),
    x="planta",
    y="tasa_cumplimiento",
    text="tasa_cumplimiento",
    title="Cumplimiento normativo por planta (%)",
)
fig.update_traces(texttemplate="%{text:.2f}%", textposition="outside")
fig.update_layout(yaxis_title="Cumplimiento (%)", xaxis_title="Planta")
fig.show()

In [ ]:
fig = px.bar(
    resumen_planta.sort_values("alertas", ascending=False),
    x="planta",
    y="alertas",
    text="alertas",
    title="Registros con alerta operativa por planta",
)
fig.update_traces(textposition="outside")
fig.update_layout(yaxis_title="Cantidad de alertas", xaxis_title="Planta")
fig.show()

## 6. Comportamiento de DBO y eficiencia

Estas visualizaciones ayudan a revisar si los incumplimientos se concentran en fechas, plantas o rangos de caudal especificos.

In [ ]:
fig = px.line(
    df.sort_values("fecha_registro"),
    x="fecha_registro",
    y="DBO_salida_mg_L",
    color="planta",
    title="Evolucion de DBO de salida por planta",
)
fig.update_layout(xaxis_title="Fecha", yaxis_title="DBO salida (mg/L)")
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="caudal_entrada_m3_d",
    y="DBO_salida_mg_L",
    color="estado_cumplimiento",
    symbol="planta",
    hover_data=["fecha_registro", "eficiencia_remocion_DBO_%", "alerta_operativa"],
    title="Caudal de entrada vs DBO de salida",
)
fig.update_layout(xaxis_title="Caudal entrada (m3/d)", yaxis_title="DBO salida (mg/L)")
fig.show()

In [ ]:
fig = px.box(
    df,
    x="planta",
    y="eficiencia_remocion_DBO_%",
    color="planta",
    points="all",
    title="Distribucion de eficiencia de remocion de DBO por planta",
)
fig.update_layout(xaxis_title="Planta", yaxis_title="Eficiencia de remocion DBO (%)", showlegend=False)
fig.show()

In [ ]:
fig = px.scatter(
    df,
    x="energia_aeracion_kWh",
    y="eficiencia_remocion_DBO_%",
    color="planta",
    hover_data=["fecha_registro", "estado_cumplimiento", "alerta_operativa"],
    title="Energia de aireacion vs eficiencia de remocion de DBO",
)
fig.update_layout(xaxis_title="Energia aireacion (kWh)", yaxis_title="Eficiencia de remocion DBO (%)")
fig.show()

## 7. Valores atipicos

Se usa z-score sobre variables numericas. Los registros marcados con 1 superan el umbral absoluto de 3 desviaciones estandar.

In [ ]:
columnas_anomalia = [col for col in df.columns if col.startswith("anomalia_")]

resumen_anomalias = (
    df[columnas_anomalia]
    .sum()
    .rename("cantidad_anomalias")
    .reset_index()
    .rename(columns={"index": "variable"})
)
resumen_anomalias["variable"] = resumen_anomalias["variable"].str.replace("anomalia_", "", regex=False)
resumen_anomalias = resumen_anomalias.sort_values("cantidad_anomalias", ascending=False)

resumen_anomalias

In [ ]:
registros_anomalos = df[df[columnas_anomalia].sum(axis=1) > 0].copy()
registros_anomalos[
    ["fecha_registro", "planta", "DBO_salida_mg_L", "lodos_generados_kg_d", "eficiencia_remocion_DBO_%"] + columnas_anomalia
]

## 8. Archivos de salida

Se generan los archivos que consume el dashboard y los entregables por area:

- `outputs/operaciones_aqualimpia.csv`
- `outputs/gestion_ambiental_aqualimpia.csv`
- `outputs/resumen_indicadores.csv`
- `outputs/calidad_datos.csv`
- `outputs/dataset_procesado.joblib`

In [ ]:
operaciones, gestion_ambiental, resumen_indicadores = generar_archivos_salida(df)

archivos_generados = sorted(path.name for path in OUTPUT_PATH.glob("*"))
archivos_generados

In [ ]:
resumen_indicadores

## 9. Registros con alerta operativa

La tabla siguiente muestra los registros que requieren atencion por incumplimiento normativo o baja eficiencia del tratamiento.

In [ ]:
alertas = df[df["alerta_operativa"] == "Alerta"].copy()

alertas[[
    "fecha_registro",
    "planta",
    "caudal_entrada_m3_d",
    "DBO_entrada_mg_L",
    "DBO_salida_mg_L",
    "eficiencia_remocion_DBO_%",
    "energia_aeracion_kWh",
    "lodos_generados_kg_d",
    "estado_cumplimiento",
]].sort_values(["planta", "fecha_registro"]).head(20)

## 10. Conclusiones

El analisis muestra una situacion operacional que requiere seguimiento: aunque la eficiencia promedio de remocion de DBO es alta, el cumplimiento normativo global es bajo. Esto indica que la eficiencia por si sola no basta para explicar el desempeno ambiental, y conviene revisar con mayor detalle los registros con DBO de salida elevada.

Planta Norte aparece como el foco mas critico por su menor tasa de cumplimiento, mientras que Planta Centro concentra tambien una cantidad alta de alertas. Las anomalias detectadas en lodos generados y DBO de salida deben revisarse como posibles eventos operativos puntuales o registros que requieren validacion.